# Document Question Answering System (RAG)

A Retrieval-Augmented Generation pipeline in 7 phases:

1. Document Ingestion
2. Sentence-aware Text Chunking
3. Embedding Creation
4. Vector Database (FAISS) with per-document source tags
5. Query Processing
6. Hybrid Retrieval
7. Answer Generation (local flan-t5-base)



## 0. Setup

In [ ]:
!pip install -q sentence-transformers faiss-cpu pypdf transformers torch rank_bm25

In [ ]:
import os
import re
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## 1. Document Ingestion

Documents such as PDFs or text files are loaded and converted into raw text.

In [ ]:
def load_document(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        reader = PdfReader(path)
        pages = [page.extract_text() or "" for page in reader.pages]
        text = "\n".join(pages)
    elif ext == ".txt":
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
    else:
        raise ValueError(f"Unsupported file type: {ext}. Use .pdf or .txt")

    text = re.sub(r"\s+", " ", text).strip()
    return text

### Upload your own document

Upload a PDF


In [ ]:
from google.colab import files
uploaded = files.upload()
DOC_PATHS = list(uploaded.keys())
print("Uploaded:", DOC_PATHS)

Saving Sun_Tzu_1000_Word_Essay.pdf to Sun_Tzu_1000_Word_Essay.pdf
Uploaded: ['Sun_Tzu_1000_Word_Essay.pdf']


## 2. Text Chunking (sentence-aware)

The text is split into smaller chunks to improve retrieval accuracy.

In [ ]:
def split_sentences(text: str):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sentences if s]


def chunk_text(text: str, chunk_size: int = 80, overlap_sentences: int = 2):
    sentences = split_sentences(text)
    chunks = []
    current, current_len = [], 0

    for sentence in sentences:
        sentence_len = len(sentence.split())

        if current and current_len + sentence_len > chunk_size:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences > 0 else []
            current_len = sum(len(s.split()) for s in current)

        current.append(sentence)
        current_len += sentence_len

    if current:
        chunks.append(" ".join(current))

    return chunks

In [ ]:
sample_chunks = chunk_text(load_document(DOC_PATHS[0]), chunk_size=60, overlap_sentences=1)
print(f"{DOC_PATHS[0]} split into {len(sample_chunks)} chunks")
for i, c in enumerate(sample_chunks):
    print(f"--- chunk {i} ({len(c.split())} words) ---")
    print(c[:150], "...\n")

Sun_Tzu_1000_Word_Essay.pdf split into 26 chunks
--- chunk 0 (55 words) ---
Sun Tzu is remembered as one of history's most influential thinkers on strategy. Although many details of his life remain uncertain, his ideas have sh ...

--- chunk 1 (57 words) ---
The Art of War emphasizes careful planning, understanding both oneself and one's opponent, using resources wisely, and avoiding unnecessary conflict.  ...

--- chunk 2 (59 words) ---
It encourages leaders to gather information, remain adaptable, and make decisions based on changing circumstances instead of rigid rules. The text exp ...

--- chunk 3 (56 words) ---
Flexibility is presented as a key quality because conditions constantly change. Sun Tzu compares effective strategy to water, which naturally adjusts  ...

--- chunk 4 (56 words) ---
Beyond warfare, readers have applied these principles to business, sports, education, negotiation, and everyday life. Companies study competitive stra ...

--- chunk 5 (42 words) ---
Instead o

## 3 & 4. Embedding Creation+Vector Database

Each chunk is embedded with `all-MiniLM-L6-v2` and stored in a FAISS index.
`VectorStore` also tracks which source file each chunk came from, so
multiple documents can be indexed together and answers can be traced back to
their source.

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")


class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatL2(dim)
        self.chunks = []
        self.sources = []

    def add(self, embeddings, chunks, source):
        self.index.add(embeddings)
        self.chunks.extend(chunks)
        self.sources.extend([source] * len(chunks))


store = None
for path in DOC_PATHS:
    text = load_document(path)
    doc_chunks = chunk_text(text, chunk_size=60, overlap_sentences=1)
    embeddings = embed_model.encode(doc_chunks, convert_to_numpy=True)

    if store is None:
        store = VectorStore(dim=embeddings.shape[1])

    store.add(embeddings, doc_chunks, source=os.path.basename(path))
    print(f"Indexed {len(doc_chunks)} chunks from {path}")

print(f"Total chunks indexed: {len(store.chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 26 chunks from Sun_Tzu_1000_Word_Essay.pdf
Total chunks indexed: 26


## 5&6. Hybrid Retrieval +Re-ranking

Retrieval combines two signals instead of relying on embeddings alone:

- **BM25** — classic keyword search, good at catching exact terms.
- **Vector search** — FAISS similarity search, good at catching semantic
  matches that don't share exact wording.

The two rankings are merged with Reciprocal Rank Fusion (RRF)

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

tokenized_corpus = [c.lower().split() for c in store.chunks]
bm25 = BM25Okapi(tokenized_corpus)


def hybrid_retrieve(question, store, embed_model, bm25, reranker,
                     top_k=3, fusion_pool=15, rrf_k=60):
    n = len(store.chunks)
    pool_size = min(fusion_pool, n)


    tokenized_query = question.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_rank = list(np.argsort(bm25_scores)[::-1])


    q_embedding = embed_model.encode([question], convert_to_numpy=True)
    _, indices = store.index.search(q_embedding, n)
    vector_rank = [int(i) for i in indices[0] if i != -1]


    fused_scores = {}
    for rank, idx in enumerate(bm25_rank):
        fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(vector_rank):
        fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    fused_ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    candidate_idx = [idx for idx, _ in fused_ranked[:pool_size]]
    candidates = [
        {"text": store.chunks[i], "source": store.sources[i]}
        for i in candidate_idx
    ]


    if candidates:
        pairs = [[question, c["text"]] for c in candidates]
        scores = reranker.predict(pairs)
        candidates = [c for _, c in sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)]

    return candidates[:top_k]


question = "Who is sun tzu?"
retrieved_chunks = hybrid_retrieve(question, store, embed_model, bm25, reranker, top_k=3)

for i, c in enumerate(retrieved_chunks):
    print(f"Retrieved chunk {i} (from {c['source']}):\n{c['text']}\n")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieved chunk 0 (from Sun_Tzu_1000_Word_Essay.pdf):
Sun Tzu is remembered as one of history's most influential thinkers on strategy. Although many details of his life remain uncertain, his ideas have shaped military planning, leadership, business, politics, and personal development for centuries. The Art of War emphasizes careful planning, understanding both oneself and one's opponent, using resources wisely, and avoiding unnecessary conflict.

Retrieved chunk 1 (from Sun_Tzu_1000_Word_Essay.pdf):
Sun Tzu is remembered as one of history's most influential thinkers on strategy. Although many details of his life remain uncertain, his ideas have shaped military planning, leadership, business, politics, and personal development for centuries. The Art of War emphasizes careful planning, understanding both oneself and one's opponent, using resources wisely, and avoiding unnecessary conflict.

Retrieved chunk 2 (from Sun_Tzu_1000_Word_Essay.pdf):
Sun Tzu is remembered as one of history's mo

## 7. Answer Generation

Uses `flan t5 base`,A language model generates an answer using the retrieved context.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def build_prompt(question, context_chunks):

    context = "\n\n".join(f"[Source: {c['source']}]\n{c['text']}" for c in context_chunks)
    return (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def generate_answer(question, context_chunks):
    prompt = build_prompt(question, context_chunks)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


answer = generate_answer(question, retrieved_chunks)
print("Q:", question)
print("A:", answer)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Q: Who is sun tzu?
A: thinkers on strategy


In [ ]:
class RAGSystem:
    def __init__(self, chunk_size=60, overlap_sentences=1, top_k=3):
        self.chunk_size = chunk_size
        self.overlap_sentences = overlap_sentences
        self.top_k = top_k
        self.store = None
        self.bm25 = None

    def build_index(self, doc_paths):
        if isinstance(doc_paths, str):
            doc_paths = [doc_paths]

        for path in doc_paths:
            text = load_document(path)
            doc_chunks = chunk_text(text, self.chunk_size, self.overlap_sentences)
            embeddings = embed_model.encode(doc_chunks, convert_to_numpy=True)

            if self.store is None:
                self.store = VectorStore(dim=embeddings.shape[1])

            self.store.add(embeddings, doc_chunks, source=os.path.basename(path))
            print(f"Indexed {len(doc_chunks)} chunks from {path}")

        tokenized_corpus = [c.lower().split() for c in self.store.chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def add_documents(self, doc_paths):
        self.build_index(doc_paths)

    def ask(self, question):
        retrieved = hybrid_retrieve(question, self.store, embed_model, self.bm25, reranker, top_k=self.top_k)
        answer = generate_answer(question, retrieved)
        return {"question": question, "answer": answer, "context_used": retrieved}


rag = RAGSystem()
rag.build_index(DOC_PATHS)

for q in [
    "Who is sun tzu?",
    "What is art of war?",
    "was sun tzu a female?",
]:
    result = rag.ask(q)
    print("Q:", result["question"])
    print("A:", result["answer"])
    for c in result["context_used"]:
        print(f"   (from {c['source']})")
    print()

Indexed 26 chunks from Sun_Tzu_1000_Word_Essay.pdf
Q: Who is sun tzu?
A: thinkers on strategy
   (from Sun_Tzu_1000_Word_Essay.pdf)
   (from Sun_Tzu_1000_Word_Essay.pdf)
   (from Sun_Tzu_1000_Word_Essay.pdf)

Q: What is art of war?
A: The Art of War emphasizes careful planning, understanding both oneself and one's opponent, using resources wisely, and avoiding unnecessary conflict.
   (from Sun_Tzu_1000_Word_Essay.pdf)
   (from Sun_Tzu_1000_Word_Essay.pdf)
   (from Sun_Tzu_1000_Word_Essay.pdf)

Q: was sun tzu a female?
A: no
   (from Sun_Tzu_1000_Word_Essay.pdf)
   (from Sun_Tzu_1000_Word_Essay.pdf)
   (from Sun_Tzu_1000_Word_Essay.pdf)



## Conclusion

This notebook covers the full RAG loop end to end: loading a document,
sentence-aware chunking, embedding, storing in a FAISS index, hybrid
BM25 + vector retrieval fused with Reciprocal Rank Fusion, cross-encoder
re-ranking for precision, and answer generation with a local flan-t5 mode